# Text Preprocessing for VOSviewer — NLTK Pipeline

**Corpus:** 98,681 articles, Brazilian authors, Life and Health Sciences (Scopus)  
**Processed columns:** `title` and `description`  
**Preserved columns:** `eid`, `score<hic>`, `score<hic lmic>`, `score<lmic>`  
**Output:** 4 CSV files for VOSviewer import (total, HIC, LMIC, HIC-LMIC)

---

## Corpus issues identified during prior diagnosis

1. **Corrupted encoding:** The file (latin-1/cp1252) contains conversion artefacts:
   - `O~` replacing the apostrophe (`patients'` becomes `patientsO~`)
   - `E^` replacing a space in multi-word expressions (`in vitro` becomes `inE^vitro`)
   - `p<0.05` encoded as `pE^=E^0.05` (statistical values)
   - Total: ~60,000 affected tokens

2. **Out-of-domain contamination** (corpus spans all Life Sciences in Scopus):
   - Agriculture / ecology: 16% of articles
   - Animal models: 9.7%
   - Materials science: 7%

3. **Multilingual leakage:** 9.1% of articles contain Portuguese / Spanish words

4. **Critical polysemy flagged by reviewers:** `film` appears 1,060 times


## Cell 1 — Installation

Install all dependencies in one step.

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "nltk", "pandas", "tqdm", "wordfreq", "-q"],
    check=True,
)

import nltk
for resource in ("stopwords", "wordnet", "omw-1.4"):
    nltk.download(resource, quiet=True)

print("Dependencies ready.")


## Cell 2 — Configuration

Set `INPUT_FILE` and `OUTPUT_DIR` here before running.

In [ ]:
import os

# ---- user-configurable ------------------------------------------------
INPUT_FILE = "titulos_resumos.csv"   # path to the Scopus corpus CSV
OUTPUT_DIR = "."                     # directory for all output files
# -----------------------------------------------------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Input : {os.path.abspath(INPUT_FILE)}")
print(f"Output: {os.path.abspath(OUTPUT_DIR)}")


## Cell 3 — Imports and stopword lists

In [ ]:
import re
import json
from collections import Counter, defaultdict

import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm.notebook import tqdm

tqdm.pandas()
lemmatizer = WordNetLemmatizer()

# Base NLTK English stopwords
_STOPWORDS_NLTK: set = set(stopwords.words("english"))

# Bibliometric stopwords: generic abstract-structure terms with no thematic signal
_STOPWORDS_BIBLIOMETRIC: set = {
    "study", "studies", "paper", "papers", "article", "review",
    "result", "results", "conclusion", "conclusions", "finding", "findings",
    "background", "objective", "objectives", "aim", "aims", "purpose",
    "introduction", "method", "methods", "approach", "approaches",
    "discussion", "context",
    "high", "low", "higher", "lower", "increase", "decrease",
    "increased", "decreased", "significantly", "significant",
    "associated", "compared", "observed", "showed", "reported",
    "found", "performed", "evaluated", "assessed", "analyzed",
    "identified", "investigated", "obtained", "measured", "collected",
    "presented", "described", "revealed", "indicated",
    "data", "datum", "analysis", "analyses", "model", "models",
    "sample", "samples", "group", "groups", "control", "case", "cases",
    "total", "number", "rate", "rates", "level", "levels", "value",
    "values", "mean", "range", "index", "factor", "factors", "test",
    "type", "types", "period", "year", "years", "month", "months",
    "area", "areas", "region", "regions", "population", "populations",
    "large", "small", "single", "different", "similar", "common",
    "important", "potential", "major", "primary", "overall", "main",
    "current", "new", "first", "two", "three", "four", "five",
    "respectively", "therefore", "however", "although", "whereas",
    "present", "work", "provide", "suggest", "show", "indicate",
}

# Multilingual leakage stopwords (PT/ES structured-abstract headers)
_STOPWORDS_MULTILINGUAL: set = {
    "objetivo", "objetivos", "metodo", "metodos", "resultado", "resultados",
    "conclusao", "conclusoes", "introducao", "discussao", "antecedentes",
    "pacientes", "foram", "para", "como", "mais", "entre", "este", "esta",
    "esse", "essa", "pelo", "pela", "pelos", "pelas", "numa", "neste",
    "estudio", "fueron", "conclusion", "conclusiones",
}

STOPWORDS: set = _STOPWORDS_NLTK | _STOPWORDS_BIBLIOMETRIC | _STOPWORDS_MULTILINGUAL

# Monitor terms: short tokens preserved as analytically informative anchor terms.
# Cite them explicitly in the paper (not treated as stopwords).
MONITOR_TERMS: set = {
    "who",   # World Health Organization  (~8,170 articles)
    "lmic",  # Low- and middle-income countries (~312 articles)
    "hic",   # High-income countries (~237 articles)
    "sus",   # Sistema Unico de Saude (~140 articles)
}

print(f"Monitor terms : {MONITOR_TERMS}")
print(
    f"Stopwords     : {len(_STOPWORDS_NLTK)} NLTK "
    f"+ {len(_STOPWORDS_BIBLIOMETRIC)} bibliometric "
    f"+ {len(_STOPWORDS_MULTILINGUAL)} multilingual "
    f"= {len(STOPWORDS)} total"
)


## Cell 4 — Disambiguation thesaurus

Built from a prior diagnostic of the real corpus (98,681 articles). Every rule carries an explicit rationale for methodological documentation (Van Eck & Waltman, 2010).

**Principle:** no term is discarded. Polysemous terms are *specified* when accompanied by a disambiguating qualifier; the compound form (underscore-joined) is treated as a distinct node by VOSviewer. A bare term without its qualifier is left intact.

**Strategies:**
- **specification** -- qualifier + polysemous term => unambiguous canonical form
- **normalisation** -- spelling variants and acronyms => single canonical form

Each rule is a 4-tuple: `(regex_pattern, replacement, rationale, strategy)`.  
**Order matters:** longer expressions must precede shorter ones to prevent partial substitutions.


In [ ]:
# fmt: off
# Each entry: (regex_pattern, replacement, rationale, strategy)
# strategy in {"specification", "normalisation"}

THESAURUS: list = [

    # =========================================================================
    # BLOCK 1 -- SPECIFICATION OF POLYSEMOUS TERMS
    # Direct response to reviewer comment (Van Eck & Waltman, 2010).
    # =========================================================================

    # --- film (1,060x in corpus) ---
    # Senses: polymer film, biofilm, thin film, documentary film.
    # Bare "film" without qualifier is left intact.
    (r"\bpolymer\s+film(?:s)?\b",    "polymer_film",
     "polymer film (materials science)", "specification"),
    (r"\bthin\s+film(?:s)?\b",       "thin_film",
     "thin film (physics/engineering)", "specification"),
    (r"\bprotective\s+film(?:s)?\b", "protective_film",
     "protective film (materials)", "specification"),
    (r"\bbiofilm(?:s)?\b",           "biofilm",
     "biofilm (already unambiguous -- normalises spelling)", "specification"),

    # --- resistance ---
    (r"\bdrug\s+resistance\b",          "drug_resistance",          "drug resistance", "specification"),
    (r"\bantibiotic\s+resistance\b",    "antibiotic_resistance",    "antibiotic resistance", "specification"),
    (r"\bantimicrobial\s+resistance\b", "antimicrobial_resistance", "antimicrobial resistance", "specification"),
    (r"\binsulin\s+resistance\b",       "insulin_resistance",       "insulin resistance", "specification"),
    (r"\bchemotherapy\s+resistance\b",  "chemotherapy_resistance",  "chemotherapy resistance", "specification"),

    # --- membrane ---
    (r"\bcell\s+membrane(?:s)?\b",     "cell_membrane",     "cell membrane", "specification"),
    (r"\bplasma\s+membrane(?:s)?\b",   "plasma_membrane",   "plasma membrane", "specification"),
    (r"\bmucous\s+membrane(?:s)?\b",   "mucous_membrane",   "mucous membrane", "specification"),
    (r"\bbasement\s+membrane(?:s)?\b", "basement_membrane", "basement membrane", "specification"),
    (r"\bdialysis\s+membrane(?:s)?\b", "dialysis_membrane", "dialysis membrane (biomedical engineering)", "specification"),

    # --- matrix ---
    (r"\bextracellular\s+matrix\b",           "extracellular_matrix",      "extracellular matrix", "specification"),
    (r"\bmatrix\s+metalloproteinase(?:s)?\b",  "matrix_metalloproteinase",  "matrix metalloproteinase", "specification"),
    (r"\bbone\s+matrix\b",                    "bone_matrix",               "bone matrix", "specification"),
    (r"\bnuclear\s+matrix\b",                 "nuclear_matrix",            "nuclear matrix", "specification"),

    # --- fiber / fibre ---
    (r"\bnerve\s+fib(?:er|re)(?:s)?\b",    "nerve_fiber",    "nerve fiber", "specification"),
    (r"\bmuscle\s+fib(?:er|re)(?:s)?\b",   "muscle_fiber",   "muscle fiber", "specification"),
    (r"\bdietary\s+fib(?:er|re)(?:s)?\b",  "dietary_fiber",  "dietary fiber", "specification"),
    (r"\bcollagen\s+fib(?:er|re)(?:s)?\b", "collagen_fiber", "collagen fiber", "specification"),

    # --- layer ---
    (r"\bepithelial\s+layer(?:s)?\b", "epithelial_layer",        "epithelial layer", "specification"),
    (r"\bcorneal\s+layer(?:s)?\b",    "corneal_layer",           "corneal layer", "specification"),
    (r"\blayer[\s-]by[\s-]layer\b",  "layer_by_layer_assembly", "layer-by-layer (nanotechnology)", "specification"),

    # --- agent ---
    (r"\bcytotoxic\s+agent(?:s)?\b",        "chemotherapy_drug", "cytotoxic agent", "specification"),
    (r"\bchemotherapeutic\s+agent(?:s)?\b", "chemotherapy_drug", "chemotherapeutic agent", "specification"),
    (r"\bantineoplastic\s+agent(?:s)?\b",   "chemotherapy_drug", "antineoplastic agent", "specification"),
    (r"\binfectious\s+agent(?:s)?\b",       "infectious_agent",  "infectious agent", "specification"),
    (r"\bcausative\s+agent(?:s)?\b",        "causative_agent",   "causative agent", "specification"),
    (r"\bcontrast\s+agent(?:s)?\b",         "contrast_agent",    "contrast agent (diagnostic imaging)", "specification"),

    # --- screening ---
    (r"\bcancer\s+screening\b",   "cancer_screening",   "cancer screening", "specification"),
    (r"\bneonatal\s+screening\b", "neonatal_screening", "neonatal screening", "specification"),
    (r"\bnewborn\s+screening\b",  "neonatal_screening", "newborn screening -> neonatal_screening", "specification"),
    (r"\bdrug\s+screening\b",     "drug_screening",     "drug screening", "specification"),
    (r"\bgenetic\s+screening\b",  "genetic_screening",  "genetic screening", "specification"),

    # --- burden ---
    (r"\bdisease\s+burden\b",   "disease_burden",   "disease burden", "specification"),
    (r"\bfinancial\s+burden\b", "financial_burden", "financial burden", "specification"),
    (r"\beconomic\s+burden\b",  "financial_burden", "economic burden -> financial_burden", "specification"),
    (r"\btumor\s+burden\b",     "tumor_burden",     "tumor burden", "specification"),

    # --- synthesis ---
    (r"\bprotein\s+synthesis\b",  "protein_synthesis",  "protein synthesis", "specification"),
    (r"\bDNA\s+synthesis\b",      "dna_synthesis",      "DNA synthesis", "specification"),
    (r"\bcollagen\s+synthesis\b", "collagen_synthesis", "collagen synthesis", "specification"),

    # --- surface ---
    (r"\bbody\s+surface\s+area\b",   "body_surface_area", "body surface area", "specification"),
    (r"\bmucosal\s+surface(?:s)?\b",  "mucosal_surface",   "mucosal surface", "specification"),

    # =========================================================================
    # BLOCK 2 -- NORMALISATION: SPELLING VARIANTS AND ACRONYMS
    # Full forms must precede acronyms so that acronym rules only fire on
    # tokens not already replaced.
    # =========================================================================

    # --- WHO / World Health Organization ---
    # ~8,170 articles total. Short token "who" avoids VOSviewer concatenation.
    (r"\bworld\s+health\s+organ(?:ization|isation)s?\b", "who",
     "world health organization -> who", "normalisation"),
    (r"\bWHO\b", "who", "WHO -> who", "normalisation"),

    # --- LMIC / Low- and middle-income countries ---
    (r"\blow[\s-]and[\s-]middle[\s-]income\s+countr(?:y|ies)\b", "lmic",
     "low- and middle-income country -> lmic", "normalisation"),
    (r"\bdeveloping\s+countr(?:y|ies)\b", "lmic",
     "developing country -> lmic", "normalisation"),
    (r"\blow[\s-]resource\s+setting(?:s)?\b", "lmic",
     "low-resource setting -> lmic", "normalisation"),
    (r"\bresource[\s-]limited\s+setting(?:s)?\b", "lmic",
     "resource-limited setting -> lmic", "normalisation"),
    (r"\bLMIC(?:s)?\b", "lmic", "LMIC -> lmic", "normalisation"),

    # --- HIC / High-income countries ---
    (r"\bhigh[\s-]income\s+countr(?:y|ies)\b", "hic",
     "high-income country -> hic", "normalisation"),
    (r"\bHIC(?:s)?\b", "hic", "HIC -> hic", "normalisation"),

    # --- SUS (Sistema Unico de Saude) ---
    (r"\bSUS\b", "sus", "SUS -> sus", "normalisation"),

    # --- NCD / Noncommunicable disease ---
    (r"\bnon[\s-]?communicable\s+disease(?:s)?\b", "noncommunicable_disease",
     "noncommunicable disease -> noncommunicable_disease", "normalisation"),
    (r"\bNCD(?:s)?\b", "noncommunicable_disease", "NCD -> noncommunicable_disease", "normalisation"),

    # --- SDG / Sustainable Development Goal ---
    (r"\bsustainable\s+development\s+goal(?:s)?\b", "sustainable_development_goal",
     "sustainable development goal -> sustainable_development_goal", "normalisation"),
    (r"\bSDG(?:s)?\b", "sustainable_development_goal", "SDG -> sustainable_development_goal", "normalisation"),

    # --- UHC / Universal Health Coverage ---
    (r"\buniversal\s+health\s+coverage\b", "universal_health_coverage",
     "universal health coverage -> universal_health_coverage", "normalisation"),
    (r"\bUHC\b", "universal_health_coverage", "UHC -> universal_health_coverage", "normalisation"),

    # --- British vs. American English spelling ---
    (r"\btumour(?:s)?\b",         "tumor",      "tumour -> tumor",           "normalisation"),
    (r"\bleukaemia\b",            "leukemia",   "leukaemia -> leukemia",     "normalisation"),
    (r"\banaemia\b",              "anemia",     "anaemia -> anemia",         "normalisation"),
    (r"\bbehaviour(?:s)?\b",      "behavior",   "behaviour -> behavior",     "normalisation"),
    (r"\bprogramme(?:s)?\b",      "program",    "programme -> program",      "normalisation"),
    (r"\bcentre(?:s)?\b",         "center",     "centre -> center",          "normalisation"),
    (r"\bfoetus(?:es)?\b",        "fetus",      "foetus -> fetus",           "normalisation"),
    (r"\bfoetal\b",               "fetal",      "foetal -> fetal",           "normalisation"),
    (r"\bhaemoglobin\b",          "hemoglobin", "haemoglobin -> hemoglobin", "normalisation"),
    (r"\bhaematolog(?:y|ical)\b", "hematology", "haematology -> hematology", "normalisation"),
    (r"\boestrogen\b",            "estrogen",   "oestrogen -> estrogen",     "normalisation"),
    (r"\banaesthesia\b",          "anesthesia", "anaesthesia -> anesthesia", "normalisation"),
    (r"\bpaediatric(?:s)?\b",     "pediatric",  "paediatric -> pediatric",   "normalisation"),
    (r"\bgynaecolog(?:y|ical)\b", "gynecology", "gynaecology -> gynecology", "normalisation"),
]
# fmt: on

_strategy_counts = Counter(s for *_, s in THESAURUS)
print(f"Thesaurus loaded: {len(THESAURUS)} rules")
for strategy, n in sorted(_strategy_counts.items()):
    print(f"  {strategy}: {n} rules")


## Cell 5 — Language filter

Removes tokens that are neither English nor recognised scientific terms.

**Three-layer logic:**
1. Token contains `_` (thesaurus compound) => pass through unchanged.
2. Token in English dictionary (`wordfreq` top-100k) => keep.
3. Token in `SCIENTIFIC_EXCEPTIONS` => keep.
4. Anything else => discard.

`SCIENTIFIC_EXCEPTIONS` was built automatically from the 450 most frequent corpus tokens absent from the English dictionary; every entry was manually verified as valid scientific vocabulary.


In [ ]:
from wordfreq import top_n_list

EN_DICT: set = set(top_n_list("en", 100_000))
print(f"English dictionary loaded: {len(EN_DICT):,} words")

SCIENTIFIC_EXCEPTIONS: set = {
    # Pathogens and parasites
    "cruzi", "leishmania", "schistosoma", "leptospira", "brucei", "brucella",
    "rickettsia", "ehrlichia", "burkholderia", "corynebacterium", "cryptococcus",
    "paracoccidioides", "sporothrix", "candida", "neoformans", "capsulatum",
    "gattii", "fumigatus", "abscessus", "pseudotuberculosis", "candidatus",
    "enterococcus", "faecalis", "gingivalis", "typhimurium", "enterica",
    "bradyrhizobium", "xanthomonas", "colletotrichum", "trichoderma",
    "metarhizium", "brachiaria", "meloidogyne", "rhizosphere",
    # Tropical diseases and vectors
    "leishmaniasis", "chagasic", "trypanosomes", "trypomastigotes",
    "promastigotes", "amastigotes", "triatoma", "rhodnius", "prolixus",
    "lutzomyia", "longipalpis", "mansoni", "infantum", "amazonensis",
    "braziliensis", "brasiliensis", "donovani", "bothrops",
    "amblyomma", "rhipicephalus", "microplus", "sanguineus", "albopictus",
    # Viruses
    "zikv", "denv", "chikv", "htlv",
    # Species / taxonomy
    "sapajus", "guianensis", "angustifolia", "coffea", "citri", "indicus",
    "oryzae", "solani", "lactis", "mutans", "tabaci", "absoluta",
    "frugiperda", "anastrepha", "nelore", "bovine", "inulin", "phaseolus",
    "passiflora", "myrcia", "araucaria", "fabaceae", "asteraceae", "myrtaceae",
    "anura", "acari", "stricto", "abortus",
    # Brazilian biomes and ecological terms
    "caatinga", "restinga", "neotropics", "cerrado", "pantanal",
    "biogeographic", "biogeographical", "endemism", "sympatric",
    "monophyletic", "monophyly", "phylogeographic", "phylogenies",
    "intraspecific", "conspecific", "ontogenetic", "ontogeny", "allometric",
    "successional", "nestedness", "herbivory", "mutualistic", "parasitoid",
    "parasitoids", "endophytic", "congeners", "transect", "transects",
    "floristic", "phenological", "bromeliad", "bromeliads", "stingless",
    "macrophyte", "macrophytes", "arbuscular", "edaphic", "oxisol",
    "biochar", "lignocellulosic", "bagasse", "cowpea",
    "eucalypt", "herbage", "postharvest", "barcoding", "multilocus",
    "microsatellites", "heterozygosity", "introgression", "admixed",
    "karyotypes", "rflp", "genotyped", "genotypic", "exome",
    "gwas", "qtls", "missense", "dsrna",
    # Laboratory methods and bioinformatics
    "qpcr", "taqman", "maldi", "dpph", "abts", "tbars", "naocl",
    "pyrosequencing", "metagenomic", "transcriptomic", "proteomic",
    "metabolomics", "metagenomics", "morphometric", "morphometry",
    "morphologic", "ultrastructural", "ultrastructure", "histomorphometric",
    "histopathologic", "cytological", "electromyography", "electromyographic",
    "echocardiographic", "angiographic", "absorptiometry", "spirometry",
    "isokinetic", "submaximal", "genbank", "scielo", "cinahl", "embase",
    "psycinfo", "clinicaltrials", "cbct", "ivus", "lllt", "rtms",
    "photodynamic", "intravitreal", "intraperitoneal", "intraoral",
    "intraepithelial", "intraclass", "interobserver", "interquartile",
    "purinergic", "immunostaining", "immunoreactivity", "immunosorbent",
    "immunomodulatory", "immunogenic", "bioassay", "bioassays", "bioactivity",
    "biocompatibility", "biocontrol", "seropositive", "seroprevalence",
    "serotypes", "serovar", "serogroup", "serology", "parasitemia",
    "parasitological", "antiparasitic",
    # Statistics and epidemiology
    "tukey", "kruskal", "wilcoxon", "bonferroni", "cronbach", "weibull",
    "confounders", "multicentre", "randomisation", "sociodemographic",
    "hrqol", "ohrqol", "dalys", "birthweight", "mvpa", "rmse", "dmft",
    "heterologous", "subscales", "accuracies",
    # Pharmacology and oncology
    "trastuzumab", "bevacizumab", "imatinib", "everolimus", "bortezomib",
    "docetaxel", "tenofovir", "efavirenz", "benznidazole", "amphotericin",
    "chlorhexidine", "fluconazole", "rivaroxaban", "ticagrelor", "denosumab",
    "haart", "neoadjuvant", "antiproliferative", "antinociceptive",
    "antiplatelet", "genotoxic", "genotoxicity",
    "phytochemical", "phenolics", "polyphenol", "anthocyanin", "caffeic",
    "chlorogenic", "rutin", "carrageenan", "pilocarpine", "apomorphine",
    "streptozotocin", "bradykinin", "kinin", "cathepsin", "arginase",
    "peptidase", "glucosidase", "phytase", "xylanase", "lipases",
    "myeloperoxidase", "malondialdehyde", "thiobarbituric",
    # Molecular and cell biology
    "cxcl", "cxcr", "ppar", "ampk", "rankl", "foxp", "smad", "trpv",
    "trpa", "nlrp", "inos", "hmga", "hmgb", "anxa", "bohv", "treg",
    "pbmc", "pbmcs", "vegfr", "fgfr", "mmps", "timp", "hnscc",
    "oscc", "cabg", "stemi", "nafld", "gvhd", "hsct",
    "inflammasome", "metalloproteinase", "metalloproteinases", "annexin",
    "galectin", "trophoblast", "granulosa", "luteal", "estrous", "preantral",
    "ovariectomized", "adiponectin", "adiposity", "dyslipidemia", "steatosis",
    "cardiometabolic", "cardiorespiratory", "baroreflex", "hemodynamics",
    "hyperalgesia", "allodynia", "nociception", "nociceptive", "neuroinflammation",
    "microglial", "glutamatergic", "striatal", "dorsolateral", "angiogenic",
    "neovascularization", "osteoblast", "osteoclast", "osteoclasts",
    "osteogenic", "osteotomy", "osseointegration", "hydroxyapatite",
    "cardiomyocyte", "cardiomyocytes", "eosinophil", "supernatants",
    "overexpressing", "downregulated", "vitrification", "cryopreserved",
    "glcnac", "glycolytic", "ethanolic", "diphenyl", "pufa",
    # Clinical specialties
    "endodontic", "periapical", "odontogenic", "malocclusion", "mesial",
    "condylar", "masticatory", "demineralization", "microhardness", "microtensile",
    "silane", "dentine", "implantitis", "edentulous",
    "keratoconus", "choroidal", "bcva", "orofacial",
    "patellofemoral", "meniscal", "femoris", "soleus", "gastrocnemius",
    "vastus", "handgrip", "sarcopenia", "acromegaly",
    "inspiratory", "expiratory", "ventilatory", "normotensive",
    "lvef", "anodal", "dlpfc", "crohn", "mucositis", "antral",
    "clinicopathological", "pathophysiological", "etiologic", "etiological",
    "virologic", "preoperatively", "postoperatively",
    "multiparous", "parturition", "pubertal",
    # Ecology / animal health
    "ruminal", "monensin", "crossbred", "gavage",
    "mycotoxin", "sublethal", "entomopathogenic", "bacteriocin",
    "mycobacterial", "helminth", "midgut",
    # Brazilian toponyms used as analytical outcomes
    "tocantins", "pelotas",
    # Databases and journals
    "karger", "liebert",
    # Other specialised terms
    "septate", "ascospores", "redescription", "redescribed", "nomenclatural",
    "dissimilarity", "biphasic", "modulatory", "molecularly",
}

print(f"Scientific exceptions: {len(SCIENTIFIC_EXCEPTIONS):,} terms")
print()
print("Language-filter logic:")
print("  token with '_' (thesaurus compound) => pass through unchanged")
print("  token in EN_DICT                     => keep")
print("  token in SCIENTIFIC_EXCEPTIONS       => keep")
print("  anything else                        => discard")


## Cell 6 — Preprocessing functions

In [ ]:
# Substitution log: {eid: [substitution_record, ...]}
# Declared at module level; cleared explicitly before each full run (Cell 9)
# to prevent accumulation across repeated executions.
substitution_log: dict = defaultdict(list)


def fix_encoding(text: str) -> str:
    """Repair latin-1/cp1252 encoding artefacts found in the corpus.

    Known artefacts and their corrections:
    - O~/o~ in place of apostrophe  (patientsO~ -> patients)
    - E^/e^ in place of space       (inE^vitro  -> in vitro)
    - Eth    in place of hyphen
    - N~     in place of em-dash
    - o^/O^/_ replacing quotation marks or control characters
    """
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[\xd5\xf5]s\b", "s", text)   # patientsO~ -> patients
    text = re.sub(r"[\xd5\xf5]\b",  "",  text)   # alzheimerO~s -> alzheimers
    text = re.sub(r"[\xca\xea]",     " ", text)   # inE^vitro -> in vitro
    text = re.sub(r"[\xd0]",          "-", text)   # corrupted hyphen
    text = re.sub(r"[_\xf4\xd4]",    " ", text)   # corrupted quotation marks
    text = re.sub(r"[\xd1]",          " ", text)   # corrupted em-dash
    return text


def clean_text(text: str) -> str:
    """Structural cleaning: strip HTML, remove references, normalise whitespace.

    Applied before thesaurus substitution so that patterns match clean text.
    """
    if not isinstance(text, str):
        return ""
    text = fix_encoding(text)
    text = re.sub(r"<[^>]+>",             " ", text)  # HTML/XML tags
    text = re.sub(r"\[\d+\]",          " ", text)  # in-text citations [1]
    text = re.sub(r"\(\d{4}\)",        " ", text)  # bare years (2019)
    text = re.sub(r"p\s*[=<>]\s*0?\.\d+", " ", text)  # p-values: p<0.05
    text = re.sub(r"\b\d+\.?\d*\s*%", " ", text)  # percentages: 42.3%
    text = re.sub(r"[^a-zA-Z0-9\s_]",    " ", text)  # keep letters, digits, _
    text = re.sub(r"\s+",                " ", text).strip()
    return text.lower()


def apply_thesaurus(text: str, eid: str, field: str) -> str:
    """Apply THESAURUS rules in order, recording every substitution.

    Parameters
    ----------
    text : str
        Pre-cleaned lowercase text.
    eid : str
        Article identifier used to key the substitution log.
    field : str
        Source field name ("title" or "description").

    Returns
    -------
    str
        Text after all matching thesaurus rules have been applied.
    """
    for pattern, replacement, rationale, strategy in THESAURUS:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        if matches:
            substitution_log[eid].append({
                "field":       field,
                "pattern":     pattern,
                "replacement": replacement,
                "strategy":    strategy,
                "occurrences": len(matches),
                "rationale":   rationale,
            })
            text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text


def lemmatize_and_filter(text: str) -> list:
    """Tokenise, lemmatise (WordNetLemmatizer), and filter tokens.

    Compound tokens produced by the thesaurus (containing "_") bypass
    lemmatisation and pass through directly.

    Two-pass lemmatisation (noun then verb) follows VOSviewer preprocessing
    recommendations (Van Eck & Waltman, 2010).
    """
    tokens = re.findall(r"\b[a-z][a-z_]+[a-z]\b", text)  # minimum 3 chars
    result = []
    for token in tokens:
        if "_" in token:  # thesaurus compound: pass through
            result.append(token)
            continue
        lemma = lemmatizer.lemmatize(token, pos="n")  # noun pass
        lemma = lemmatizer.lemmatize(lemma,  pos="v") # verb pass
        if lemma in STOPWORDS or len(lemma) <= 2:
            continue
        if lemma not in EN_DICT and lemma not in SCIENTIFIC_EXCEPTIONS:
            continue
        result.append(lemma)
    return result


def preprocess(text: str, eid: str, field: str) -> str:
    """Full preprocessing pipeline for a single text field.

    Steps: encoding repair -> structural cleaning -> thesaurus substitution
    -> lemmatisation -> stopword / language filtering.

    Returns a space-separated token string for direct VOSviewer corpus-file import.
    """
    text = clean_text(text)
    text = apply_thesaurus(text, eid, field)
    tokens = lemmatize_and_filter(text)
    return " ".join(tokens)


print("Preprocessing functions loaded.")


## Cell 7 — Load and validate corpus

In [ ]:
df = pd.read_csv(INPUT_FILE, encoding="latin-1", sep=None, engine="python")
df.columns = df.columns.str.strip()
df["title"]       = df["title"].fillna("")
df["description"] = df["description"].fillna("")

print(f"Articles loaded : {len(df):,}")
print(f"Columns         : {list(df.columns)}")
print()

for grp, col in [("HIC", "score<hic>"), ("LMIC", "score<lmic>"), ("HIC-LMIC", "score<hic lmic>")]:
    n = (df[col] == 1).sum()
    print(f"  {grp:<10}: {n:>7,} articles ({100 * n / len(df):.1f}%)")

# Confirm mutual exclusivity of the three score groups
overlap = (
    (df["score<hic>"]      == 1).astype(int)
  + (df["score<lmic>"]     == 1).astype(int)
  + (df["score<hic lmic>"] == 1).astype(int)
  > 1
).sum()
print(f"\nArticles in more than one group: {overlap}  (expected: 0)")


## Cell 8 — Sample test

Run this cell before the full run to validate the pipeline on 20 articles.


In [ ]:
_sample = df.sample(20, random_state=42)

for _, row in _sample.head(5).iterrows():
    eid = str(row["eid"])
    title_clean = preprocess(row["title"], eid, "title")
    desc_clean  = preprocess(row["description"], eid, "description")
    print(f"\nEID: {eid}")
    print(f"  TITLE   (raw)  : {row['title'][:80]}")
    print(f"  TITLE   (clean): {title_clean[:80]}")
    print(f"  ABSTRACT (raw) : {str(row['description'])[:120]}")
    print(f"  ABSTRACT (cln) : {desc_clean[:120]}")

substitution_log.clear()
print("\nSample test complete. Log cleared.")


## Cell 9 — Full processing

~98,681 articles. Estimated runtime: 15-25 minutes depending on hardware.


In [ ]:
substitution_log.clear()

print("Processing titles ...")
df["title_clean"] = df.progress_apply(
    lambda row: preprocess(row["title"], str(row["eid"]), "title"), axis=1
)

print("Processing abstracts ...")
df["description_clean"] = df.progress_apply(
    lambda row: preprocess(row["description"], str(row["eid"]), "description"), axis=1
)

print(f"Done. {len(df):,} articles processed.")


## Cell 10 — Disambiguation report

This output is the **methodological evidence** requested by the reviewers.


In [ ]:
_rule_counter:    Counter = Counter()
_article_counter: Counter = Counter()

for eid, subs in substitution_log.items():
    for sub in subs:
        _rule_counter[(sub["replacement"], sub["rationale"])] += sub["occurrences"]
        _article_counter[sub["replacement"]] += 1

print("=== DISAMBIGUATION REPORT ===")
print(f"Articles with at least one substitution: {len(substitution_log):,}")
print()
print(f"{'Replacement':<35} {'Occurrences':>12} {'Articles':>10}")
print("-" * 60)

_by_replacement: dict = defaultdict(lambda: {"occ": 0, "arts": 0})
for (replacement, _rationale), occ in _rule_counter.items():
    _by_replacement[replacement]["occ"]  += occ
    _by_replacement[replacement]["arts"]  = _article_counter[replacement]

for repl, vals in sorted(_by_replacement.items(), key=lambda x: -x[1]["occ"]):
    print(f"{repl:<35} {vals['occ']:>12,} {vals['arts']:>10,}")

# Specific verification for "film" (reviewer-flagged term)
_film_entries = [
    (eid, s)
    for eid, subs in substitution_log.items()
    for s in subs
    if "film" in s["pattern"].lower()
]
_unique_film_eids = len({eid for eid, _ in _film_entries})
print(f"\nVerification 'film': {len(_film_entries):,} occurrences in {_unique_film_eids:,} articles.")


## Cell 11 — Top terms by subgroup

In [ ]:
subgroups: dict = {
    "TOTAL":    df,
    "HIC":      df[df["score<hic>"] == 1],
    "LMIC":     df[df["score<lmic>"] == 1],
    "HIC-LMIC": df[df["score<hic lmic>"] == 1],
}

for name, subdf in subgroups.items():
    combined = " ".join(subdf["title_clean"] + " " + subdf["description_clean"])
    freq = Counter(combined.split()).most_common(20)
    print(f"\n=== TOP 20 -- {name} ({len(subdf):,} articles) ===")
    for term, count in freq:
        print(f"  {term}: {count:,}")


## Cell 12 — Export datasets for VOSviewer

### Expected VOSviewer input format

VOSviewer accepts two text-import modes:

**Option A -- Corpus file (recommended):**  
CSV with a `label` column (document identifier) and a `text` column (free text).  
Menu: *Create -> Create a map based on text data -> Corpus file*

**Option B -- Bibliographic data (Web of Science / Scopus native export):**  
Not applicable here (file is not in the native Scopus export format).

All four datasets are exported as Option A.


In [ ]:
score_cols = [c for c in df.columns if "score" in c.lower()]
df["text_vos"] = df["title_clean"] + " " + df["description_clean"]

# Four mutually exclusive subsets (confirmed in Cell 7):
#   total    -- all 98,681 articles (overall Brazilian output profile)
#   hic      -- articles led by high-income-country authors
#   lmic     -- articles led by low/middle-income-country authors
#   hic_lmic -- articles with HIC + LMIC co-leadership
datasets: dict = {
    "total":    df,
    "hic":      df[df["score<hic>"] == 1],
    "lmic":     df[df["score<lmic>"] == 1],
    "hic_lmic": df[df["score<hic lmic>"] == 1],
}

print(f"Saving files to: {os.path.abspath(OUTPUT_DIR)}")
print()

for name, subdf in datasets.items():
    vos = subdf[["eid", "text_vos"]].copy()
    vos.columns = ["label", "text"]
    fpath = os.path.join(OUTPUT_DIR, f"vosviewer_{name}.csv")
    vos.to_csv(fpath, index=False, encoding="utf-8")
    print(f"  vosviewer_{name}.csv       -> {len(vos):>7,} articles")

# Audit file: raw and cleaned fields side by side for manual spot-checks
audit_cols = ["eid", "title", "title_clean", "description", "description_clean"] + score_cols
audit_path = os.path.join(OUTPUT_DIR, "corpus_preprocessed_full.csv")
df[audit_cols].to_csv(audit_path, index=False, encoding="utf-8")
print(f"  corpus_preprocessed_full.csv -> {os.path.abspath(audit_path)}")

# Per-article substitution log (methodological evidence for reviewers)
log_path = os.path.join(OUTPUT_DIR, "thesaurus_log.json")
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(dict(substitution_log), f, ensure_ascii=False, indent=2)
print(f"  thesaurus_log.json           -> {os.path.abspath(log_path)}")

print("\nAll files saved.")


## Cell 13 — How to import in VOSviewer

1. Open VOSviewer.
2. *Create -> Create a map based on text data.*
3. Select: **Corpus file**.
4. Load: `vosviewer_total.csv` (or `hic` / `lmic` / `hic_lmic`).
5. Verify: *label column* = `label`, *text column* = `text`.
6. Counting method: **Binary** (recommended for co-occurrence).
7. Minimum number of occurrences -- adjust to corpus size:
   - Total (98k articles): minimum 10-20.
   - LMIC (~8k articles): minimum 5-10.
8. Repeat for each of the four files.


## Cell 14 — Methodological summary table (for manuscript)

In [ ]:
print("=== PREPROCESSING METHODOLOGY TABLE ===")
print()
print(f"{'Step':<25} {'Tool':<28} {'Decision / Rationale'}")
print("-" * 90)
_rows = [
    ("Encoding repair",     "Regex (Python)",           "latin-1 artefacts: O~->apostrophe, E^->space"),
    ("Structural cleaning", "Regex (Python)",           "Remove HTML, citations, p-values, percentages"),
    ("Disambiguation",      "Custom thesaurus",         f"{len(THESAURUS)} rules, 2 strategies (Van Eck & Waltman, 2010)"),
    ("  Specification",     "",                         "Ambiguous term + qualifier -> canonical compound"),
    ("  Normalisation",     "",                         "Spelling variants and acronyms -> standard form"),
    ("Stopwords",           "NLTK + custom",            f"{len(_STOPWORDS_NLTK)} NLTK + bibliometric + multilingual = {len(STOPWORDS)} total"),
    ("Lemmatisation",       "NLTK WordNetLemmatizer",   "Two passes: noun (pos=n) then verb (pos=v)"),
]
for step, tool, rationale in _rows:
    print(f"  {step:<23} {tool:<28} {rationale}")

print()
print("Out-of-domain contamination addressed:")
for domain, pct in [("Materials science", "7.0"), ("Animal models", "9.7"), ("Agriculture/ecology", "16.0")]:
    print(f"  {domain:<22} {pct}% of corpus")

print()
print("Output datasets:")
for name, subdf in datasets.items():
    print(f"  vosviewer_{name}.csv : {len(subdf):,} articles")

print()
print("Auditable evidence: thesaurus_log.json -- per-article substitution record (keyed by EID)")
